In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# BFloat16 Adder Module

This notebook incrementally shows how to build a floating point addition circuit for the bfloat16 format.  

The design is inspired by [this paper](https://ieeexplore.ieee.org/abstract/document/10440496). 

## Floating point addition algorithm

### Stage 1: Input Parsing and Extraction
- **Objective**: Extract the basic components (sign, exponent, mantissa) from the input numbers.
- **Steps**:
  1. Extract the sign bits from both input numbers.
  2. Extract the exponent bits from both input numbers.
  3. Extract the mantissa bits, including the hidden implicit bit.

### Stage 2: Exponent Difference Calculation
- **Objective**: Align the exponents of the two numbers.
- **Steps**:
  1. Subtract the exponent of the second input from the first to find the offset.
  2. Determine which number has the larger exponent.
  3. Align the mantissa of the number with the smaller exponent by right shifting.

### Stage 3: Mantissa Alignment and Bit Preparation
- **Objective**: Prepare mantissas for addition.
- **Steps**:
  1. Apply the necessary right shifts to the smaller mantissa for alignment.
  2. Generate Sticky, Guard, and Round (SGR) bits to facilitate rounding.

### Stage 4: Mantissa Addition/Subtraction
- **Objective**: Add or subtract the aligned mantissas.
- **Steps**:
  1. Perform addition or subtraction based on the sign bits.
  2. Handle negative results by inverting if necessary.
  3. Use a Leading Zero Detector (LZD) to find leading zeros in the result.

### Stage 5: Normalization and Final Adjustments
- **Objective**: Finalize the result by normalization and rounding.
- **Steps**:
  1. Normalize the result using the LZD output to adjust the exponent.
  2. Implement rounding and correct for any overflow by adjusting the exponent.
  3. Determine the final sign of the result.

### Block Diagram of 5 stage FP32 Adder

<img src="./assets/float-adder.png" width="500">

# PyRTL Implementation

### Imports & Setup

In [155]:
from IPython.display import display_svg
from typing import List, Tuple
import pyrtl
from pyrtl.rtllib.libutils import twos_comp_repr, rev_twos_comp_repr
from pyrtl import (
    WireVector, 
    Const, 
    Input,
    Output, 
    Register, 
    Simulation, 
    SimulationTrace, 
    reset_working_block
)
from kai.src.utils import custom_render_trace
from kai.src.bfloat16 import BF16

In [141]:
pyrtl.set_debug_mode(True)

In [89]:
E_BITS  = 8
M_BITS  = 7
MSB     = E_BITS + M_BITS

### Repr Funcs

In [292]:
def repr_bf16(x):
    binary = format(x, "016b")
    bf16_value = BF16(x, True)
    return f"{binary[:1]}.{binary[1:9]}.{binary[9:]} ({bf16_value.tensor:.3f})"

def repr_sign(x):
    return "0 (+)" if x == 0 else "1 (-)"

def repr_exp(x):
    return f"{format(x, '08b')} (unbiased={x - 127})"

def repr_mantissa(x):
    binary = format(x, "08b")
    decimal = int(binary[0]) + int(binary[1:], 2) / (2**7)
    return f"{binary[:1]}.{binary[1:]} ({decimal:.3f})"

def repr_ext_mantissa(x):
    binary = format(x, "016b")
    decimal = int(binary[0]) + int(binary[1:], 2) / (2**15)
    return f"{binary[:1]}.{binary[1:]} ({decimal:.3f})"

def repr_num(x):
    return format(x, '0b') + f" ({x})"

In [267]:
format(0b100000000, '08b')

'100000000'

## Stage 1: Data Extraction

In [19]:
def extract_sign(input_a: WireVector, input_b: WireVector, msb: int) -> Tuple[WireVector, WireVector]:
    sign_a = WireVector(1, name='sign_a')
    sign_b = WireVector(1, name='sign_b')
    
    sign_a <<= input_a[msb]  # Assuming the MSB is the sign bit
    sign_b <<= input_b[msb]
    
    return sign_a, sign_b

In [20]:
def extract_exponent(input_a: WireVector, input_b: WireVector, e_bits: int) -> Tuple[WireVector, WireVector]:
    exp_a = WireVector(e_bits, name='exp_a')
    exp_b = WireVector(e_bits, name='exp_b')
    
    exp_a <<= input_a[-(1+e_bits):-1]
    exp_b <<= input_b[-(1+e_bits):-1]
    
    return exp_a, exp_b

In [21]:
def extract_mantissa(input_a: WireVector, input_b: WireVector, m_bits: int) -> Tuple[WireVector, WireVector]:
    mantissa_a = WireVector(m_bits+1, name='mantissa_a')
    mantissa_b = WireVector(m_bits+1, name='mantissa_b')
    
    # Concatenate the implicit leading 1
    mantissa_a <<= pyrtl.concat(Const(1, 1), input_a[:m_bits])
    mantissa_b <<= pyrtl.concat(Const(1, 1), input_b[:m_bits])
    
    return mantissa_a, mantissa_b

In [22]:
def stage_1(input_a: WireVector, input_b: WireVector, e_bits: int, m_bits: int):
    # Extract components
    sign_a, sign_b = extract_sign(input_a, input_b, e_bits + m_bits)
    exp_a, exp_b = extract_exponent(input_a, input_b, e_bits)
    mantissa_a, mantissa_b = extract_mantissa(input_a, input_b, m_bits)
    return sign_a, sign_b, exp_a, exp_b, mantissa_a, mantissa_b

In [82]:
def test_stage1(e_bits=E_BITS, m_bits=M_BITS):
    reset_working_block()
    total_bits = 1 + e_bits + m_bits
    # Inputs should match the bfloat16 format: 1 sign bit, 8 exponent bits, and 7 mantissa bits
    input_a = Input(total_bits, name='input_a')
    input_b = Input(total_bits, name='input_b')

    # Extract components
    stage_1(input_a, input_b, e_bits, m_bits)
    
    # Simulate and test
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    sim.step({'input_a': 0b0100000001000000, 'input_b': 0b0100000010000000})  # Example inputs

    input_repr_map = {
        'input_a': repr_bf16,
        'input_b': repr_bf16,
        'sign_a': repr_sign,
        'sign_b': repr_sign,
        'exp_a': repr_exp,
        'exp_b': repr_exp,
        'mantissa_a': repr_mantissa,
        'mantissa_b': repr_mantissa,
    }
    trace_list = list(input_repr_map.keys())

    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_stage1()


<IPython.core.display.Javascript object>

## Stage 2: Exponent Subtraction and Detection

In [124]:
def calculate_exponent_difference(exp_a: WireVector, exp_b: WireVector):
    assert len(exp_a) == len(exp_b)
    exp_diff = WireVector(len(exp_a)+1, name='exp_diff')
    exp_greater = WireVector(len(exp_a), name='exp_greater')

    exp_diff <<= pyrtl.signed_sub(exp_a, exp_b)  # This can be negative, indicating which is larger
    exp_greater <<= pyrtl.mux(exp_diff[8], exp_a, exp_b)
    return exp_diff, exp_greater

In [245]:
def stage_2_test(
    exp_a: WireVector, 
    exp_b: WireVector, 
    mant_a: WireVector,
    mant_b: WireVector,
    e_bits: int, 
    m_bits: int
):
    assert len(exp_a) == len(exp_b) == e_bits
    assert len(mant_a) == len(mant_b) == m_bits + 1

    exp_diff = WireVector(e_bits+1, name='exp_diff')
    shift_amount = WireVector(e_bits, name='shift_amount')
    exp_larger = WireVector(e_bits+1, name='exp_larger')
    mant_smaller = WireVector(m_bits+1, name='mant_smaller')
    mant_larger = WireVector(m_bits+1, name='mant_larger')

    exp_diff <<= exp_a - exp_b  # This can be negative, indicating which is larger
    exp_larger <<= pyrtl.mux(exp_diff[e_bits], exp_a, exp_b)
    shift_amount <<= pyrtl.mux(exp_diff[e_bits], exp_diff[:e_bits], ~exp_diff[:e_bits] + 1)

    # Select the smaller mantissa to align and the larger to keep unchanged
    mant_smaller <<= pyrtl.mux(exp_diff[e_bits], mant_b, mant_a)
    mant_larger <<= pyrtl.mux(exp_diff[e_bits], mant_a, mant_b)

    return exp_larger, shift_amount, mant_smaller, mant_larger

In [246]:
def test_stage_2():
    reset_working_block()

    exp_a, exp_b = Input(E_BITS, 'exp_a'), Input(E_BITS, 'exp_b')
    mant_a, mant_b = Input(M_BITS+1, 'mant_a'), Input(M_BITS+1, 'mant_b')

    exp_larger, shift_amount, mant_smaller, mant_larger = stage_2_test(exp_a, exp_b, mant_a, mant_b, E_BITS, M_BITS)

    exp_larger_out = Output(E_BITS, 'exp_larger_out')
    shift_amount_out = Output(E_BITS+1, 'shift_amount_out')
    mant_smaller_out = Output(M_BITS+1, 'mant_smaller_out')
    mant_larger_out = Output(M_BITS+1, 'mant_larger_out')

    exp_larger_out <<= exp_larger
    shift_amount_out <<= shift_amount
    mant_smaller_out <<= mant_smaller
    mant_larger_out <<= mant_larger
    
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)

    inputs = {
        'exp_a': [1, 128, 129, 255],
        'exp_b': [128, 255, 128, 200],
        'mant_a': [129, 133, 200, 192],
        'mant_b': [160, 192, 160, 148]
    }
    sim.step_multiple(inputs)  # Example inputs

    # Representation maps for visualization
    input_repr_map = {
        'exp_a': repr_exp,
        'exp_b': repr_exp,
        'mant_a': repr_mantissa,
        'mant_b': repr_mantissa,
        'exp_larger': repr_exp,
        'exp_diff': lambda x: rev_twos_comp_repr(x, E_BITS+1),
        'shift_amount': repr_num,
        'mant_smaller': repr_mantissa,
        'mant_larger': repr_mantissa,
    }
    trace_list = list(input_repr_map.keys())

    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map, repr_func=repr_num)

test_stage_2()

<IPython.core.display.Javascript object>

#### Extras

In [85]:
def align_mantissa(mantissa_a, mantissa_b, exp_diff):
    aligned_mantissa_a = pyrtl.WireVector(len(mantissa_a), name='aligned_mantissa_a')
    aligned_mantissa_b = pyrtl.WireVector(len(mantissa_b), name='aligned_mantissa_b')
    
    # Shift mantissas based on exponent difference
    shift_amount = pyrtl.abs(exp_diff)  # Ensure non-negative shift amount
    
    aligned_mantissa_a <<= pyrtl.mux(exp_diff >= 0, mantissa_a, mantissa_a >> shift_amount)
    aligned_mantissa_b <<= pyrtl.mux(exp_diff < 0, mantissa_b, mantissa_b >> shift_amount)
    
    return aligned_mantissa_a, aligned_mantissa_b

In [86]:
def stage_2(exp_a, exp_b, mantissa_a, mantissa_b):
    exp_diff, larger_exp = calculate_exponent_difference(exp_a, exp_b)
    aligned_mantissa_a, aligned_mantissa_b = align_mantissa(mantissa_a, mantissa_b, exp_diff)
    return exp_diff, larger_exp, aligned_mantissa_a, aligned_mantissa_b


In [87]:
def test_stage2(e_bits=E_BITS, m_bits=M_BITS):
    pyrtl.reset_working_block()
    total_bits = 1 + e_bits + m_bits
    input_a = pyrtl.Input(total_bits, name='input_a')
    input_b = pyrtl.Input(total_bits, name='input_b')

    # Stage 1 outputs
    sign_a, sign_b, exp_a, exp_b, mantissa_a, mantissa_b = stage_1(input_a, input_b, e_bits, m_bits)

    # Stage 2 processing
    exp_diff, larger_exp, aligned_mantissa_a, aligned_mantissa_b = stage_2(exp_a, exp_b, mantissa_a, mantissa_b)

    # Outputs for testing
    exp_diff_out = pyrtl.Output(name='exp_diff')
    larger_exp_out = pyrtl.Output(name='larger_exp')
    aligned_mantissa_a_out = pyrtl.Output(name='aligned_mantissa_a')
    aligned_mantissa_b_out = pyrtl.Output(name='aligned_mantissa_b')

    exp_diff_out <<= exp_diff
    larger_exp_out <<= larger_exp
    aligned_mantissa_a_out <<= aligned_mantissa_a
    aligned_mantissa_b_out <<= aligned_mantissa_b

    # Simulate and test
    sim_trace = pyrtl.SimulationTrace()
    sim = pyrtl.Simulation(tracer=sim_trace)
    sim.step({'input_a': 0b0100000001000000, 'input_b': 0b0100000010000000})  # Example inputs

    # Representation maps for visualization
    input_repr_map = {
        'input_a': repr_bf16,
        'input_b': repr_bf16,
        'sign_a': repr_sign,
        'sign_b': repr_sign,
        'exp_a': repr_exp,
        'exp_b': repr_exp,
        'mantissa_a': repr_mantissa,
        'mantissa_b': repr_mantissa,
        'exp_diff': str,
        'larger_exp': repr_exp,
        'aligned_mantissa_a': repr_mantissa,
        'aligned_mantissa_b': repr_mantissa,
    }
    trace_list = list(input_repr_map.keys())

    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

test_stage2()


AttributeError: module 'pyrtl' has no attribute 'abs'

## Stage 3: Mantissa Alignment and SGR Generator

### Align Mantissa

In [310]:
def align_mantissa(
    mant_smaller: WireVector,
    shift_amount: WireVector,
    m_bits: int
) -> WireVector:
    """
    Align the smaller mantissa by shifting right
    
    Args:
        mant_smaller: mantissa to be shifted (m_bits + 1 wide)
        shift_amount: number of positions to shift right (e_bits wide)
        m_bits: number of mantissa bits (7 for bfloat16)
    
    Returns:
        aligned_mantissa_msb, aligned_mantissa_lsb: shifted mantissa
    """
    assert len(mant_smaller) == m_bits + 1
    assert len(shift_amount) == E_BITS
    
    # Detect if shift amount exceeds mantissa width
    max_useful_shift = 2 * (m_bits + 1)
    
    # Clamp shift amount to maximum useful value
    clamped_shift = WireVector(E_BITS, 'clamped_shift')
    clamped_shift <<= pyrtl.select(
        shift_amount > max_useful_shift,
        max_useful_shift,
        shift_amount,
    )
    
    # Perform the right shift
    extended_mantissa = pyrtl.concat(mant_smaller, pyrtl.Const(0, m_bits+1))

    aligned_mantissa = WireVector(max_useful_shift, name='aligned_mantissa')
    assert len(extended_mantissa) == len(aligned_mantissa)

    aligned_mantissa <<= pyrtl.shift_right_logical(extended_mantissa, clamped_shift)
    return pyrtl.chop(aligned_mantissa, *[m_bits+1]*2)

def test_align_mantissa():
    reset_working_block()
    
    mant_smaller = Input(M_BITS + 1, 'mant_smaller')
    shift_amount = Input(E_BITS, 'shift_amount')
    
    aligned_msb, aligned_lsb = align_mantissa(mant_smaller, shift_amount, M_BITS)
    aligned_msb_out = Output(M_BITS+1, 'aligned_msb_out')
    aligned_lsb_out = Output(M_BITS+1, 'aligned_lsb_out')
    aligned_msb_out <<= aligned_msb
    aligned_lsb_out <<= aligned_lsb

    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    inputs = {
        'mant_smaller': [0b10000000, 0b10100000, 0b11000000, 0b10000000],
        'shift_amount': [1, 9, 3, 45]
    }
    
    sim.step_multiple(inputs)
    input_repr_map = {
        'mant_smaller': repr_mantissa,
        'shift_amount': repr_num,
        'clamped_shift': str,
        'aligned_mantissa': repr_ext_mantissa,
        'aligned_msb_out': repr_mantissa,
        'aligned_lsb_out': lambda x: format(x, f'0{M_BITS+1}b'),
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("Testing align_mantissa:")
test_align_mantissa()

Testing align_mantissa:


<IPython.core.display.Javascript object>

### SGR Generator

In [304]:
def generate_sgr(
    aligned_mant_lsb: WireVector,
    m_bits: int
) -> tuple[WireVector, WireVector, WireVector]:
    """
    Generate sticky, guard, and round bits
    
    Args:
        mant_smaller: original mantissa before shifting (m_bits + 1 wide)
        shift_amount: number of positions shifted right (e_bits wide)
        m_bits: number of mantissa bits (7 for bfloat16)
    
    Returns:
        sticky_bit, guard_bit, round_bit
    """
    assert len(aligned_mant_lsb) == m_bits + 1
    
    guard_bit = WireVector(1, name='guard_bit')
    round_bit = WireVector(1, name='round_bit')
    sticky_bit = WireVector(1, name='sticky_bit')
    
    guard_bit <<= aligned_mant_lsb[m_bits]
    round_bit <<= aligned_mant_lsb[m_bits-1]
    sticky_bit <<= pyrtl.or_all_bits(aligned_mant_lsb[:m_bits-1])
            
    return sticky_bit, guard_bit, round_bit

def test_generate_sgr():
    reset_working_block()
    
    aligned_mant_lsb = Input(M_BITS + 1, 'aligned_mant_lsb')
    
    sticky_bit, guard_bit, round_bit = generate_sgr(aligned_mant_lsb, M_BITS)
    
    sticky_out = Output(1, 'sticky_out')
    guard_out = Output(1, 'guard_out')
    round_out = Output(1, 'round_out')
    
    sticky_out <<= sticky_bit
    guard_out <<= guard_bit
    round_out <<= round_bit
    
    sim_trace = SimulationTrace()
    sim = Simulation(tracer=sim_trace)
    
    inputs = {
        'aligned_mant_lsb': [0b10000000, 0b01000000, 0b10100000, 0b11000000, 0b00100000],
    }
    
    sim.step_multiple(inputs)
    
    input_repr_map = {
        'aligned_mant_lsb': lambda x: format(x, f'0{M_BITS+1}b'),
        'sticky_out': repr_num,
        'guard_out': repr_num,
        'round_out': repr_num
    }
    
    trace_list = list(input_repr_map.keys())
    custom_render_trace(sim_trace, trace_list=trace_list, repr_per_name=input_repr_map)

print("\nTesting generate_sgr:")
test_generate_sgr()



Testing generate_sgr:


<IPython.core.display.Javascript object>

In [311]:
def stage_3(
    mant_smaller: WireVector,
    shift_amount: WireVector,
    m_bits: int
):
    """
    Combine alignment and SGR generation
    """
    aligned_mant_msb, aligned_mant_lsb = align_mantissa(mant_smaller, shift_amount, m_bits)
    sticky_bit, guard_bit, round_bit = generate_sgr(aligned_mant_lsb, m_bits)
    
    return aligned_mant_msb, aligned_mant_lsb, sticky_bit, guard_bit, round_bit

